# COS30049 A2 - Rebuild every result from a fresh clone, and compare with the committed tables (Kaggle CPU session)

What README sections 2-4 ask of a marker (and milestone M4 of a teammate), on a clean machine: the repository at a
fixed commit, the pinned versions, the test suite, then `python -m explorer.cli all --offline` with DS3 (Elliptic,
from the attached Kaggle dataset) and DS4 (BitcoinHeist, downloaded from UCI), the scoring methods' statistics
(`distribution`) and `figures`. `tools/compare_tables.py` then compares every rebuilt table with the committed one
(numbers within 0.01 x max(1, |committed|)); `compare.csv` lists each table's result. The rebuilt tables, figures,
small models and DS1 files are saved to commit as the frozen results. CPU only, so no GPU quota.

**Settings:** Accelerator **None**, **Internet on**. **Input:** dataset `ellipticco/elliptic-data-set`.

In [ ]:
REPO = "https://github.com/Defaul119/COS30049A2.git"
COMMIT = "0aded70f15106a9d98a2578390732e5ba492a0ef"   # the code and data this rebuild uses
# environment.yml's versions (torch is Kaggle's own, recorded; TxGPT only scores DS10 with it)
PINS = ["numpy==2.2.6", "pandas==2.2.3", "scikit-learn==1.6.1", "scipy==1.17.1", "pyarrow==20.0.0", "joblib==1.4.2",
        "matplotlib==3.10.1", "seaborn==0.13.2", "tokenizers==0.21.1", "huggingface_hub==0.30.2", "pytest==8.3.5"]

In [ ]:
import contextlib, hashlib, json, os, shutil, subprocess, sys, threading, time
from importlib.metadata import version
from pathlib import Path

T0 = time.time()                           # the session's clock: Kaggle stops a session 12 h after it starts
SRC = Path("/tmp/COS30049A2")             # the code and the data it works on; not part of the output
WORK = Path("/kaggle/working")             # the notebook's output


def sh(*cmd, log=None, cwd=None, env=None):
    """Run a command, print its output as it comes, keep a copy in WORK/<log>; raise if it fails."""
    with contextlib.ExitStack() as stack:
        copy = stack.enter_context(open(WORK / log, "a", encoding="utf-8")) if log else None
        p = stack.enter_context(subprocess.Popen(cmd, cwd=cwd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                                                 text=True, bufsize=1, env=env))
        for line in p.stdout:
            print(line, end="", flush=True)
            if copy:
                copy.write(line)
                copy.flush()
    if p.returncode:
        raise subprocess.CalledProcessError(p.returncode, cmd)


def sha256(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for block in iter(lambda: f.read(1 << 20), b""):
            h.update(block)
    return h.hexdigest()


# Kaggle grants internet (and GPUs) only to phone-verified accounts, and otherwise switches them off silently.
import socket
try:
    socket.create_connection(("github.com", 443), timeout=10).close()
except OSError:
    raise SystemExit("No internet in this session: verify a phone number (kaggle.com -> Settings -> Phone "
                     "verification), switch Internet on in the notebook's Settings panel, and run again.")

if not SRC.exists():
    sh("git", "clone", "--quiet", REPO, str(SRC))
sh("git", "checkout", "--quiet", COMMIT, cwd=SRC)
sh(sys.executable, "-m", "pip", "install", "--quiet", *PINS)
os.chdir(SRC)

def find_input(rel):
    """The file under /kaggle/input whose path ends with rel (inputs mount under their own folder names)."""
    hits = [p for p in Path("/kaggle/input").rglob(Path(rel).name) if p.as_posix().endswith(rel)]
    if not hits:
        raise SystemExit(f"{rel} is not in any input: add the inputs named at the top")
    return hits[0]


def check_base():
    """The round-1 files (dataset cos30049a2-text150m-base), each checked against the dataset's SHA256SUMS.txt."""
    sums = find_input("SHA256SUMS.txt")
    for line in sums.read_text().splitlines():
        digest, name = line.split(maxsplit=1)
        assert sha256(sums.parent / name.lstrip("*")) == digest, f"{name} differs from SHA256SUMS.txt"
    return sums.parent

In [ ]:
def cli(*args, log):
    sh(sys.executable, "-m", "explorer.cli", *args, log=log)


timing = {}
def step(name, fn):
    t = time.time()
    try:
        fn()
        timing[name] = round((time.time() - t) / 60, 1)
    except subprocess.CalledProcessError as e:
        timing[name] = f"FAILED: {e}"
    print(f"{name}: {timing[name]} min")


step("pytest", lambda: sh(sys.executable, "-m", "pytest", "-q", "-p", "no:cacheprovider", log="pytest.log"))
raw = SRC / "data" / "external" / "elliptic"
raw.mkdir(parents=True, exist_ok=True)
for name in ("elliptic_txs_features.csv", "elliptic_txs_classes.csv", "elliptic_txs_edgelist.csv"):
    shutil.copy(find_input(name), raw / name)
step("external --bitcoinheist", lambda: cli("external", "--bitcoinheist", log="external.log"))
step("all --offline", lambda: cli("all", "--offline", log="all.log"))
step("distribution", lambda: cli("distribution", log="distribution.log"))
step("figures", lambda: cli("figures", log="figures.log"))
step("compare", lambda: sh(sys.executable, "tools/compare_tables.py", "--report", str(WORK / "compare.csv"),
                           log="compare.log"))

In [ ]:
# The rebuilt results (to commit as the frozen ones), without the 263 MB DS4 model or any licensed raw rows.
for rel in ["reports/tables", "reports/figures", "reports/logs", "models/selection.json", "models/baseline.json"]:
    src = SRC / rel
    dest = WORK / "rebuilt" / rel
    if src.is_dir():
        shutil.copytree(src, dest, dirs_exist_ok=True)
    elif src.exists():
        dest.parent.mkdir(parents=True, exist_ok=True)
        shutil.copy(src, dest)
for f in sorted((SRC / "models").glob("*.joblib")):
    if f.name != "heist_best.joblib":
        (WORK / "rebuilt" / "models").mkdir(parents=True, exist_ok=True)
        shutil.copy(f, WORK / "rebuilt" / "models" / f.name)
for name in ("transactions.csv", "features.csv", "features_as_written.csv", "scores.csv"):
    dest = WORK / "rebuilt" / "data" / "processed" / "esplora"
    dest.mkdir(parents=True, exist_ok=True)
    shutil.copy(SRC / "data" / "processed" / "esplora" / name, dest / name)
(WORK / "kaggle_run.json").write_text(json.dumps({"commit": COMMIT, "minutes": timing,
    "hours": round((time.time() - T0) / 3600, 2), "python": sys.version.split()[0],
    "packages": {p: version(p) for p in ("torch", "numpy", "pandas", "scikit-learn", "matplotlib")}}, indent=2),
    encoding="utf-8")
print(json.dumps(timing, indent=2))